In [1]:
# Проверка стратификации на текущем датасете
# Запускать отдельно, ml_train.py не трогать

import pandas as pd
from sklearn.model_selection import train_test_split
from core.config import CURRENT

# Читаем сырой CSV
df = pd.read_csv(CURRENT["path"])

print("Всего строк:", len(df))
print("NaN в label:", df[CURRENT["label_col"]].isna().sum())

# Исходные доли классов
print("\nИсходные доли классов:")
print(df[CURRENT["label_col"]].value_counts(normalize=True).sort_index())

# Убираем NaN и разбиваем БЕЗ стратификации (как сейчас в load_data)
df_clean = df.dropna(subset=[CURRENT["label_col"]])
train_ns, test_ns = train_test_split(df_clean, test_size=0.2, random_state=42)

print("\n=== БЕЗ стратификации ===")
print("train:")
print(train_ns[CURRENT["label_col"]].value_counts(normalize=True).sort_index())
print("test:")
print(test_ns[CURRENT["label_col"]].value_counts(normalize=True).sort_index())

# Разбиваем СО стратификацией
train_s, test_s = train_test_split(
    df_clean, test_size=0.2, random_state=42,
    stratify=df_clean[CURRENT["label_col"]]
)

print("\n=== СО стратификацией ===")
print("train:")
print(train_s[CURRENT["label_col"]].value_counts(normalize=True).sort_index())
print("test:")
print(test_s[CURRENT["label_col"]].value_counts(normalize=True).sort_index())

# Насколько отличаются доли между train и test
print("\n=== Разница долей (train - test) ===")
diff_ns = train_ns[CURRENT["label_col"]].value_counts(normalize=True).sort_index() - \
          test_ns[CURRENT["label_col"]].value_counts(normalize=True).sort_index()
diff_s = train_s[CURRENT["label_col"]].value_counts(normalize=True).sort_index() - \
         test_s[CURRENT["label_col"]].value_counts(normalize=True).sort_index()

print("БЕЗ стратификации:")
print(diff_ns.round(4))
print("\nСО стратификацией:")
print(diff_s.round(4))

Всего строк: 439280
NaN в label: 9544

Исходные доли классов:
rating
1.0    0.814018
2.0    0.051546
3.0    0.020152
4.0    0.028550
5.0    0.045486
6.0    0.040248
Name: proportion, dtype: float64

=== БЕЗ стратификации ===
train:
rating
1.0    0.814179
2.0    0.051540
3.0    0.020155
4.0    0.028340
5.0    0.045400
6.0    0.040385
Name: proportion, dtype: float64
test:
rating
1.0    0.813376
2.0    0.051566
3.0    0.020140
4.0    0.029390
5.0    0.045830
6.0    0.039698
Name: proportion, dtype: float64

=== СО стратификацией ===
train:
rating
1.0    0.814019
2.0    0.051546
3.0    0.020152
4.0    0.028550
5.0    0.045484
6.0    0.040249
Name: proportion, dtype: float64
test:
rating
1.0    0.814015
2.0    0.051543
3.0    0.020152
4.0    0.028552
5.0    0.045493
6.0    0.040245
Name: proportion, dtype: float64

=== Разница долей (train - test) ===
БЕЗ стратификации:
rating
1.0    0.0008
2.0   -0.0000
3.0    0.0000
4.0   -0.0010
5.0   -0.0004
6.0    0.0007
Name: proportion, dtype: float

In [2]:
import sys
sys.path.append("/")

from core.data_loader import load_data
from core.clean import clean
from core.config import CURRENT

train, test = load_data()
train["Text"] = train[CURRENT["text_col"]].apply(clean)

In [3]:
y_train = train[CURRENT["label_col"]]
y_train

409322    1.0
72137     3.0
167513    4.0
426875    1.0
250946    1.0
         ... 
259178    1.0
365838    1.0
131932    1.0
146867    1.0
121958    1.0
Name: rating, Length: 351424, dtype: float64

Train time: 1.47s
Accuracy: 0.8918
F1 macro: 0.8373

   Precision  Recall      F1  Support
0     0.9605  0.9042  0.9315    69908
1     0.6674  0.8380  0.7430    16040


Train time: 41.14s
Accuracy: 0.7103
F1 macro: 0.3829

   Precision  Recall      F1  Support
1     0.9697  0.7862  0.8684    69908
2     0.1265  0.4104  0.1934     4432
3     0.1635  0.3276  0.2182     1731
4     0.1950  0.2625  0.2238     2526
5     0.3278  0.3239  0.3258     3939
6     0.4273  0.5170  0.4679     3412


In [8]:
_, test = load_data()
test

,index,rating,comment
439021,439021,NaN,"Dr. Felix Dietz ist ein toller Arzt, der einem..."
25541,25541,1.0,"Kann Frau Herbricht, die Praxis und Ihr Team n..."
338966,338966,1.0,"Fr. Dr. Saatci ist eine kompetente, nette Ärzt..."
391216,391216,1.0,Kurze Wartezeit und ausführliche Information u...
382315,382315,1.0,Schon seit vielen Jahren als Patient beim Team...
...,...,...,...
196977,196977,1.0,Angenehm freundliche neue Praxis mit nettem Pe...
384541,384541,1.0,Meine langjährigen Parodontose Beschwerden war...
65899,65899,1.0,Ich bin seit vielen Jahren Patient in der Prax...
393986,393986,1.0,"diesmal nicht die Coronakrise, sondern meine K..."


In [10]:
train.isna().sum()

index         0
rating     7660
comment       0
Text          0
dtype: int64

In [11]:
test.isna().sum()

index         0
rating     1884
comment       0
dtype: int64

In [7]:
train.Text[0]

'Ich bin franzose und bin seit ein par Wochen in muenchen. Ich hate Zahn Schmerzen und mein Kolegue hat mir Dr mainka empfohlen. Ich habe schnel ein Termin bekomen, das Team war net und meine schmerzen sind weg!! Ich bin als Angst Patient sehr zurieden!!'

In [1]:
import re

import pandas as pd
from transformers import AutoTokenizer

ModuleNotFoundError: No module named 'transformers'

In [ ]:
train = pd.read_csv("./train.csv")
train = train[train.Language == "de"]
train = train.drop(columns = "Expected")
train

In [3]:
test = pd.read_csv("./test.csv")
test = test[test.Language == "de"]
test

,Id,Language,Text
40000,1dad53cc-a052-4bd7-86b3-4695ccecdd9a,de,Wilhelm errichtete einr normannische Gentralve...
40001,54e17908-703a-4616-ac0b-a1c0cd3c0715,de,Die DPs waren 2013 also in etwa zwischen den L...
40002,823b476a-4458-4216-857e-508826b5626e,de,Das Zentrwum der Gemeinde liegt 1.300 Meter üb...
40003,9291fdfb-84f2-446f-b994-d7adf641ed25,de,Zudem war dif von Spanifn keheiu gehaltene Exi...
40004,2da27f30-8da8-4244-89dd-9b83f95fea29,de,Neben seinen Aktivitäten in der FIA-GT und ALM...
...,...,...,...
49995,b1f6e710-0a4c-4ec2-b7cf-092309f2f716,de,In einem der schwächsten Spoele der gesamten W...
49996,c37f0977-43a2-4156-bdf2-4bdd97cdbc68,de,Im Jahr 1961 wurde der Diabas noch bei Rhenegg...
49997,c81e7d0e-4d2c-4957-8774-7623594953ae,de,Hitler selbst hatte sich klar zum Privateigent...
49998,d74a117a-7a7a-4972-ba64-78206c28da7e,de,Es rind mehrere zeitgenössische Namensbezeichn...


In [4]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-german-cased")

In [5]:
?tokenizer

Signature:     
tokenizer(
    text: 'TextInput | PreTokenizedInput | list[TextInput] | list[PreTokenizedInput] | None' = None,
    text_pair: 'TextInput | PreTokenizedInput | list[TextInput] | list[PreTokenizedInput] | None' = None,
    text_target: 'TextInput | PreTokenizedInput | list[TextInput] | list[PreTokenizedInput] | None' = None,
    text_pair_target: 'TextInput | PreTokenizedInput | list[TextInput] | list[PreTokenizedInput] | None' = None,
    add_special_tokens: 'bool' = True,
    padding: 'bool | str | PaddingStrategy' = False,
    truncation: 'bool | str | TruncationStrategy | None' = None,
    max_length: 'int | None' = None,
    stride: 'int' = 0,
    is_split_into_words: 'bool' = False,
    pad_to_multiple_of: 'int | None' = None,
    padding_side: 'str | None' = None,
    return_tensors: 'str | TensorType | None' = None,
    return_token_type_ids: 'bool | None' = None,
    return_attention_mask: 'bool | None' = None,
    return_overflowing_tokens: 'bool' = False,
    

In [6]:
def tokenize_function(examples):
    return tokenizer(text = examples.Text,
                     padding = True,
                     max_length = 128,
                     truncation = True)

In [7]:
"Dezembet", "Dezember"
"zuö", "zu"
"voj" , "von"


Als Familienplanung werden Maßnahmen voj Paaren bezrichnet, die Zahl und ren Zeitpunkt der Geburt von Kindern zg p anen.

SyntaxError: invalid syntax (2323803705.py, line 6)

In [ ]:
train[train.Text.str.contains("Als Familienplanung werden Maßnahmen")].Text.values

In [ ]:
train.sample(10).Text

In [ ]:
def clean(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Убираем ссылки
    text = re.sub(r'[@#]\w+', '', text) # Убираем теги
    text = re.sub(r'(\w)\1+', r'\1', text) # Убираем повторы букв (например, "langsaaaaaam" в "langsam")
    return text

train['Text'] = train['Text'].apply(clean)
test['Text'] = test['Text'].apply(clean)
# Убираем пустые строки, которые могут появиться после очистки
train = train[train['Text'].str.strip() != '']
test = test[test['Text'].str.strip() != '']

In [8]:
from nltk.corpus import stopwords

german_stops = stopwords.words('german')

for word in ['nicht', 'kein', 'keine', 'keinen', 'keinem', 'keiner', 'keines',
             'nie', 'niemals', 'nichts', 'niemand', 'weder', 'noch',
             'kaum', 'ohne']:
    print(f"{word}: {'STOP' if word in german_stops else 'ok'}")

nicht: STOP
kein: STOP
keine: STOP
keinen: STOP
keinem: STOP
keiner: STOP
keines: STOP
nie: ok
niemals: ok
nichts: STOP
niemand: ok
weder: ok
noch: STOP
kaum: ok
ohne: STOP
